In [1]:
# =============================================================================
# Cell 1 - bootstrap and load everything the coverage computation needs.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd

iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp =pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
lad=pd.read_parquet(config.PROC_DIR/'ciciot2023_ladder_assignments.parquet')
mrec=json.loads((config.REPORTS_DIR/'ciciot2023_model_record.json').read_text())
CLASSES=mrec['classes_canonical_order']; K=len(CLASSES); FOCAL=mrec['focal_class']
c2i={c:i for i,c in enumerate(CLASSES)}; FIDX=c2i[FOCAL]
iot['y']=iot['family'].map(c2i).astype(np.int64)
PROBS_DIR=config.DATA_DIR/'ciciot_probs'
ALPHAS=[config.ALPHA_PRIMARY]+list(config.ALPHA_SENSITIVITY)
R_DRAWS=config.N_MATCHED_DRAWS
CAL_N=20000     # see cell 2 for why this size and not the eval size

# The ladder stores row_idx values taken from iot.index, and several lookups below
# index numpy arrays with them positionally. That is only valid if the frame carries a
# 0..n-1 RangeIndex, which nb31 guarantees via reset_index(drop=True). Assert it rather
# than rely on it, because a silent mismatch would misalign labels against probabilities.
assert isinstance(iot.index, pd.RangeIndex) and iot.index[0]==0 and iot.index[-1]==len(iot)-1, \
    'iot must carry a 0..n-1 RangeIndex for positional row_idx lookups to be valid'

sc_idx=iot.index[iot.partition=='source_cal_pool'].to_numpy()
tg_idx=iot.index[iot.partition=='target_pool'].to_numpy()
# dense label -> position map for the target probability array; pandas .loc on 20k labels
# inside the draw loop costs ~5 ms a call, which is minutes across 7,500 draws
TGT_POS=np.full(len(iot), -1, dtype=np.int64); TGT_POS[tg_idx]=np.arange(len(tg_idx))
YV=iot['y'].to_numpy()
y_sc=YV[sc_idx]; y_tg=YV[tg_idx]
print('classes:', CLASSES, '| focal', FOCAL, f'(idx {FIDX})')
print('src_cal_pool', len(sc_idx), '| target_pool', len(tg_idx),
      '| ladder cells', lad.groupby(["realization","rung"]).ngroups)
print('alphas', ALPHAS, '| matched draws', R_DRAWS, '| CAL_N', CAL_N)

# ---------------------------------------------------------------------------
# STALENESS GUARD. The cached probability arrays are positionally aligned to the
# partitions that existed when nb33 ran. If the split has since been rebuilt, the
# arrays no longer correspond to the current rows. When the new target pool is
# larger this surfaces as an IndexError; when it is SMALLER it would silently
# index the wrong rows and produce plausible but meaningless coverage. Check the
# shapes up front, on one file, before any work is done.
# ---------------------------------------------------------------------------
_pf=sorted(PROBS_DIR.glob('ciciot2023__*.npz'))
assert len(_pf)==30, f'expected 30 probability files, found {len(_pf)}'
_d=np.load(_pf[0]); _nt,_ns=_d['target'].shape[0], _d['srcpool'].shape[0]
print(f'\nprob files: {len(_pf)} | cached target rows {_nt:,} | cached srcpool rows {_ns:,}')
print(f'current split: target {len(tg_idx):,} | src_cal_pool {len(sc_idx):,}')
if _nt!=len(tg_idx) or _ns!=len(sc_idx):
    raise SystemExit(
        'STALE PROBABILITIES: the cached arrays were built for a different split '
        f'(target {_nt:,} vs {len(tg_idx):,}, srcpool {_ns:,} vs {len(sc_idx):,}).\n'
        'Delete data/ciciot_probs/*.npz and re-run notebook 33 before this one.\n'
        'Notebook 33 skips files that already exist, so they must be removed, not overwritten.')
assert (_d['classes'].astype(str).tolist()==CLASSES), 'cached class order differs from the model record'
print('staleness guard: cached probabilities match the current split')


Mounted at /content/drive
classes: ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web'] | focal Web (idx 7)
src_cal_pool 158079 | target_pool 456174 | ladder cells 25
alphas [0.05, 0.1, 0.2] | matched draws 10 | CAL_N 20000

prob files: 30 | cached target rows 456,174 | cached srcpool rows 158,079
current split: target 456,174 | src_cal_pool 158,079
staleness guard: cached probabilities match the current split


In [2]:
# =============================================================================
# Cell 2 - protocols, calibration-set design, and vectorised coverage.
#
# REC calibrates on D_eval itself (transductive upper bound), TSC on a labelled
# target sample, SHC on the source calibration pool. Only the calibration set
# differs; classifier, calibrator and realised scores are identical.
#
# CAL_N = 20,000 rather than the 60,000-row eval size. T_cal must carry the SAME
# focal composition as D_eval so TSC is not itself exposed to an uncontrolled
# shift. Under the Amendment 11 design the novel pool holds 4,470 target rows and
# the top rung consumes 649 for D_eval plus 216 for T_cal, so this fits with wide
# headroom. TSC and SHC use the same calibration size, so the PRIMARY contrast is
# size-matched and, at equal focal prevalence, matched per class as well.
#
# Performance: all three alphas share one sorted calibration array per class and
# one broadcast comparison, and per-class aggregation uses bincount rather than a
# Python loop. Without this the run takes hours.
# =============================================================================
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)

def aps_scores(P, rng):
    o=np.argsort(-P,axis=1); sp_=np.take_along_axis(P,o,1); cum=np.cumsum(sp_,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp_
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def mondrian_q_multi(cal_scores, y_cal, alphas, K):
    """Sort each class's true-class scores ONCE, then read off every alpha.
    Returns Q with shape (K, n_alpha) and the per-class calibration counts."""
    tc=cal_scores[np.arange(len(y_cal)), y_cal]
    Q=np.full((K,len(alphas)), np.inf); n=np.zeros(K,dtype=int)
    for k in range(K):
        s=np.sort(tc[y_cal==k]); n[k]=s.size
        for ai,a in enumerate(alphas):
            kk=int(np.ceil((s.size+1)*(1.0-a)))
            if kk<=s.size: Q[k,ai]=s[kk-1]      # identical to conformal.conformal_q
    return Q,n

def coverage_block(S_ev, y_ev, Q, n_cal, K, alphas):
    """All alphas in one broadcast; per-class sums via bincount."""
    inset = S_ev[:,:,None] <= Q[None,:,:]                 # (n, K, A)
    setsz = inset.sum(1)                                  # (n, A)
    cov   = inset[np.arange(len(y_ev)), y_ev, :]          # (n, A)
    cnt   = np.bincount(y_ev, minlength=K)
    rows=[]
    for ai,a in enumerate(alphas):
        csum = np.bincount(y_ev, weights=cov[:,ai].astype(float), minlength=K)
        ssum = np.bincount(y_ev, weights=setsz[:,ai].astype(float), minlength=K)
        for k in range(K):
            if cnt[k]==0: continue
            rows.append({'class':CLASSES[k],'alpha':a,'n_eval':int(cnt[k]),
                         'n_covered':int(csum[k]),'coverage':float(csum[k]/cnt[k]),
                         'set_size':float(ssum[k]/cnt[k]),'n_cal':int(n_cal[k]),
                         'feasible':bool(np.isfinite(Q[k,ai]))})
    return rows

# verify the fast quantile matches the canonical implementation exactly
_r=np.random.default_rng(0); _s=_r.random(500); _y=np.zeros(500,dtype=int)
_Q,_=mondrian_q_multi(_s[:,None], _y, ALPHAS, 1)
for _ai,_a in enumerate(ALPHAS):
    assert np.isclose(_Q[0,_ai], conformal_q(_s,_a)[0]), 'fast quantile diverges from conformal_q'
print('fast Mondrian quantile matches conformal.conformal_q exactly')


fast Mondrian quantile matches conformal.conformal_q exactly


In [ ]:
# =============================================================================
# Cell 3 - coverage across the ladder, the model panel and the matched draws.
# Eval sets are fixed by the ladder; both calibration sets are redrawn per matched
# draw, which is the variability this study is about.
# Per-cell index pools are precomputed once: doing the .loc inside the draw loop
# meant a 393k-row selection 7,500 times over.
# =============================================================================
files=_pf   # already validated in cell 1
fam_all=iot['family'].to_numpy(); sub_all=iot['subtype'].to_numpy()
# T_cal's focal share is derived per ladder cell FROM D_eval, not from the whole frame:
# the corrected ladder draws D_eval at the source-side focal prevalence (1.352%), not the
# frame prevalence (1.644%), and using the latter would hand T_cal a different focal share
# than D_eval, which is the uncontrolled shift T_cal exists to avoid.

cells_out=[]; skipped=0; t0=time.time()
lad_groups=list(lad.groupby(['realization','rung']))
for gi,((j,rung), g) in enumerate(lad_groups):
    ev_rows=g['row_idx'].to_numpy(); novel=set(g['novel'].iloc[0].split('|'))
    ev_pos=TGT_POS[ev_rows]; y_ev=YV[ev_rows]
    assert (ev_pos>=0).all(), 'eval rows are not all in the target pool'
    fe_sub=sub_all[ev_rows][fam_all[ev_rows]==FOCAL]
    novel_frac=float(np.isin(fe_sub, list(novel)).mean()) if fe_sub.size else 0.0
    eval_focal_share=float(fe_sub.size)/len(ev_rows)

    # ---- precompute the T_cal pools ONCE for this ladder cell ----
    avail=np.setdiff1d(tg_idx, ev_rows)
    a_fam=fam_all[avail]; a_sub=sub_all[avail]
    pool_fh=avail[(a_fam==FOCAL)&(np.isin(a_sub,list(novel)))]
    pool_fs=avail[(a_fam==FOCAL)&(~np.isin(a_sub,list(novel)))]
    pool_ot=avail[a_fam!=FOCAL]
    n_foc=int(round(CAL_N*eval_focal_share)); n_h=int(round(novel_frac*n_foc)); n_s=n_foc-n_h
    if n_h>len(pool_fh) or n_s>len(pool_fs) or (CAL_N-n_foc)>len(pool_ot):
        print(f'  realization {j} rung {rung}: T_cal quota unmet, cell skipped'); skipped+=1; continue

    for f in files:
        _,arch,sd=f.stem.split('__'); seed=int(sd.replace('seed',''))
        d=np.load(f); P_tg=d['target']; P_sc=d['srcpool']
        S_ev_base=P_tg[ev_pos]
        for draw in range(R_DRAWS):
            rng=np.random.default_rng(dseed('iotcov',j,rung,arch,seed,draw))
            s_sel=rng.choice(len(sc_idx), CAL_N, replace=False)
            t_rows=np.concatenate([rng.choice(pool_fh,n_h,replace=False),
                                   rng.choice(pool_fs,n_s,replace=False),
                                   rng.choice(pool_ot,CAL_N-n_foc,replace=False)])
            t_pos=TGT_POS[t_rows]; y_tc=YV[t_rows]
            S_ev=aps_scores(S_ev_base, np.random.default_rng(dseed('iotev',j,rung,arch,seed,draw)))
            S_sc=aps_scores(P_sc[s_sel], np.random.default_rng(dseed('iotsc',j,rung,arch,seed,draw)))
            S_tc=aps_scores(P_tg[t_pos], np.random.default_rng(dseed('iottc',j,rung,arch,seed,draw)))
            for proto,(cs,yc) in {'REC':(S_ev,y_ev),'TSC':(S_tc,y_tc),'SHC':(S_sc,y_sc[s_sel])}.items():
                Q,ncal=mondrian_q_multi(cs, yc, ALPHAS, K)
                for row in coverage_block(S_ev, y_ev, Q, ncal, K, ALPHAS):
                    row.update({'dataset':'ciciot2023','realization':int(j),'rung':float(rung),
                                'arch':arch,'seed':seed,'draw':draw,'protocol':proto,
                                'nominal':round(1-row['alpha'],3)})
                    cells_out.append(row)
    print(f'  [{gi+1}/{len(lad_groups)}] r{j} rung {rung:.2f} | rows {len(cells_out):,} | {time.time()-t0:.0f}s')
cov=pd.DataFrame(cells_out)
print(f'\ncoverage cells: {len(cov):,} | skipped: {skipped} | {time.time()-t0:.0f}s')


  [1/25] r0 rung 0.00 | rows 21,600 | 86s
  [2/25] r0 rung 0.20 | rows 43,200 | 136s
  [3/25] r0 rung 0.40 | rows 64,800 | 187s
  [4/25] r0 rung 0.60 | rows 86,400 | 236s
  [5/25] r0 rung 0.80 | rows 108,000 | 285s
  [6/25] r1 rung 0.00 | rows 129,600 | 335s
  [7/25] r1 rung 0.20 | rows 151,200 | 385s
  [8/25] r1 rung 0.40 | rows 172,800 | 434s
  [9/25] r1 rung 0.60 | rows 194,400 | 483s
  [10/25] r1 rung 0.80 | rows 216,000 | 534s
  [11/25] r2 rung 0.00 | rows 237,600 | 582s
  [12/25] r2 rung 0.20 | rows 259,200 | 631s
  [13/25] r2 rung 0.40 | rows 280,800 | 682s
  [14/25] r2 rung 0.60 | rows 302,400 | 732s
  [15/25] r2 rung 0.80 | rows 324,000 | 781s
  [16/25] r3 rung 0.00 | rows 345,600 | 831s
  [17/25] r3 rung 0.20 | rows 367,200 | 881s
  [18/25] r3 rung 0.40 | rows 388,800 | 930s
  [19/25] r3 rung 0.60 | rows 410,400 | 980s
  [20/25] r3 rung 0.80 | rows 432,000 | 1030s
  [21/25] r4 rung 0.00 | rows 453,600 | 1079s
  [22/25] r4 rung 0.20 | rows 475,200 | 1128s
  [23/25] r4 rung 0.4

In [ ]:
# =============================================================================
# Cell 4 - headline table and save.
# =============================================================================
prim=cov[np.isclose(cov.alpha, config.ALPHA_PRIMARY)]
print(f'FOCAL CLASS ({FOCAL}) COVERAGE at alpha={config.ALPHA_PRIMARY} (nominal {1-config.ALPHA_PRIMARY})')
foc=prim[prim['class']==FOCAL]
print(foc.groupby(['protocol','rung'])['coverage'].mean().unstack('rung').round(4).to_string())
print('\nfocal coverage pooled over rungs:')
print(foc.groupby('protocol')['coverage'].mean().round(4).to_string())
print('\nall classes at alpha=0.05, pooled over rungs:')
print(prim.groupby(['class','protocol'])['coverage'].mean().unstack('protocol').round(4).to_string())
print('\nmean set size by protocol:')
print(prim.groupby('protocol')['set_size'].mean().round(3).to_string())
print('\nfeasibility: cells with an infinite quantile:')
print(prim[~prim.feasible].groupby(['class','protocol']).size().to_string() or '  none')

cov.to_csv(config.REPORTS_DIR/'coverage_primary_ciciot2023.csv', index=False)
(config.REPORTS_DIR/'ciciot2023_coverage_record.json').write_text(json.dumps({
  'dataset':'ciciot2023','alphas':ALPHAS,'matched_draws':R_DRAWS,'cal_n':CAL_N,
  'eval_n_per_cell':int(lad.groupby(['realization','rung']).size().iloc[0]),
  'focal_class':FOCAL,'protocols':['REC','TSC','SHC'],
  'calibration_sizes':'TSC and SHC both calibrate on CAL_N=20,000, so the PRIMARY contrast '
      'is size-matched and, because focal prevalence is the same on both sides, matched per '
      'class too (~328 vs ~329 focal points). REC is transductive by definition and calibrates '
      'on the full evaluation set, so it uses more calibration data than the other two; it is a '
      'reference bound, not part of the primary contrast.',
  'tcal_composition':'T_cal carries the same focal held-out fraction as D_eval, so TSC is not '
      'itself exposed to an uncontrolled shift; T_cal is drawn disjoint from D_eval.',
}, indent=2, default=str))
print('\nsaved reports/coverage_primary_ciciot2023.csv and the coverage record')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb34: CIC-IoT-2023 conformal coverage under REC/TSC/SHC across the variant-holdout ladder')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
